# 04 — The estimator & its parameters

*Module 01 · k-Nearest Neighbours — notebook 4 of 6*

You have built k-NN by hand three times — the vote, the distance, the dial. That was the point: you
understand the machinery. Now you have earned the library. This notebook introduces scikit-learn's
**`KNeighborsClassifier`**, checks that it is exactly the rule you built, and then turns its knobs —
the hyperparameters that shape how k-NN behaves.

This is a different kind of notebook from the first three. Rather than one new idea, it is a guided
tour of the estimator's settings: what each does, where it breaks, and how to choose it honestly.

**Prerequisites:** notebooks 01–03 (the vote, distance and scale, the k dial), and from module 00:
notebook 05 (the decision boundary) and notebook 10 (cross-validation).

**What you will be able to do**

- Use `KNeighborsClassifier`, and confirm it reproduces the k-NN you wrote by hand.
- Tune its three main knobs: `n_neighbors`, `weights`, and `metric`.
- See where each knob helps and where it fails, and choose one by cross-validation.
- Understand how the estimator breaks an even-k tie, and why odd k is the default for two classes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

from ml_course import viz

# Fix the seed so every run tells the same story.
np.random.seed(0)
viz.use_course_style()

X, y = make_moons(n_samples=300, noise=0.30, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=0
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


# Our by-hand k-NN vote (notebooks 1-3), kept here only to check the library against it.
def knn_predict(X_ref, y_ref, queries, k=5):
    X_ref = np.asarray(X_ref, dtype=float)
    queries = np.atleast_2d(np.asarray(queries, dtype=float))
    predictions = []
    for q in queries:
        dist = np.sqrt(((X_ref - q) ** 2).sum(axis=1))
        nearest = np.argsort(dist)[:k]
        predictions.append(np.bincount(y_ref[nearest], minlength=2).argmax())
    return np.array(predictions)


print(f"training points: {X_train.shape[0]}   test points: {X_test.shape[0]}")

## From by-hand to the library

Three times now you have written the k-NN rule yourself: measure the distance to every training
point, take the k nearest, let them vote. scikit-learn's **`KNeighborsClassifier`** is that same
rule, packaged and optimized — at `fit` time it builds a spatial index (a KD-tree or ball-tree) so it
can skip most of the distance comparisons at predict time (the lazy-cost nuance from notebook 1). It
lives behind the `fit` / `predict` interface we wrapped by hand in notebook 2.

Before we trust it, two honest checks: does it predict the same labels as our by-hand version, and
does scikit-learn's `cross_val_score` reproduce the cross-validation number we computed by hand in
notebook 3?

In [ ]:
knn = KNeighborsClassifier(n_neighbors=15).fit(X_train, y_train)
sklearn_pred = knn.predict(X_test)
by_hand_pred = knn_predict(X_train, y_train, X_test, k=15)

n_diff = int((sklearn_pred != by_hand_pred).sum())
cv = cross_val_score(KNeighborsClassifier(n_neighbors=15), X_train, y_train, cv=skf)

print(f"predictions that differ (by-hand vs sklearn): {n_diff}")
print(f"sklearn test accuracy:           {(sklearn_pred == y_test).mean():.3f}")
print(f"cross_val_score (k=15, 5-fold):  {cv.mean():.3f}")

**Read the output.** Two reassurances. First, **zero** predictions differ between our by-hand k-NN
and `KNeighborsClassifier` at k = 15 — it is the very rule we built (they could differ only if two
training points were *exactly* equidistant at the edge of the neighbourhood, which essentially never
happens with real-valued coordinates), and it scores the same 0.956 on the test set. Second,
`cross_val_score` returns **0.919**, exactly the five-fold number we computed
by hand in notebook 3. The library is not a new method; it is our rule made fast, and notebook 10's
`cross_val_score` now plugs straight into it. With that settled, we can explore the knobs:
`n_neighbors`, `weights`, and `metric`.

## Knob 1 — `n_neighbors`

This is the dial from notebook 3: **k**, the number of neighbours that vote, and the bias–variance
knob. Small k overfits (jagged, memorizing); large k underfits (over-smooth); and you choose it by
cross-validation on the training data. The estimator's default is `n_neighbors=5` (our parity check
above ran it at k = 15). We studied this knob in depth in notebook 3, so here we note its place among
the others and move on.

## Knob 2 — `weights`

By default every one of the k neighbours counts equally — `weights="uniform"`. The alternative,
`weights="distance"`, scales each neighbour's vote by 1 / distance, so a point right next to the query
counts for more than one at the edge of the neighbourhood. The intuition: rather than splitting the
vote equally across the whole neighbourhood, distance-weighting tilts it toward the closer data. Let
us see what that buys as k grows.

In [ ]:
print(f"{'k':>5}  {'uniform':>9}  {'distance':>9}")
for k in (15, 51, 101, 151):
    uni = KNeighborsClassifier(n_neighbors=k, weights="uniform").fit(X_train, y_train)
    dis = KNeighborsClassifier(n_neighbors=k, weights="distance").fit(X_train, y_train)
    print(f"{k:>5}  {uni.score(X_test, y_test):>9.3f}  {dis.score(X_test, y_test):>9.3f}")

**Read the output.** At moderate k the two are close. As k grows, they diverge sharply: `uniform`
underfits badly (down to 0.678 at k = 151 — the vote is swamped by far-away points), while `distance`
holds up better (0.833) because 1 / distance weighting tilts each vote toward the nearer points, so a
large neighbourhood no longer acts as one giant equal vote. Note that it only *softens* the
over-smoothing — it does not undo it: distance at k = 151 (0.833) is still well below a well-chosen
small k (uniform k = 15 scored 0.956). It degrades gracefully, rather than fully recovering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, w in zip(axes, ("uniform", "distance"), strict=True):
    model = KNeighborsClassifier(n_neighbors=151, weights=w).fit(X_train, y_train)
    viz.plot_decision_boundary(model, X_train, y_train, resolution=200, ax=ax)
    ax.set_title(f"k = 151, weights = '{w}'")
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
fig.tight_layout()
plt.show()

**Read the figure.** Both panels use a deliberately large k = 151. On the left, `uniform` weights
have collapsed the boundary to a smooth near-diagonal sweep — 151 equal votes wash out every local
detail (this is the 0.678 from the table). On the right, `distance` weights scale each vote by
1 / distance, tilting it toward nearby points, so the boundary stays curved and close to the data
instead of collapsing to the near-diagonal sweep on the left (the 0.833 versus 0.678). Same k, same
data — only the weighting changed.

## Knob 3 — `metric` (Minkowski `p`)

How "distance" is measured is itself a parameter. The default is Euclidean (`p=2`); `p=1` is
Manhattan (notebook 2's city-block distance); other choices exist (for example Chebyshev, the largest
single-axis gap). These belong to the **Minkowski** family, parameterized by `p`. We met Euclidean
and Manhattan in notebook 2 as two ways to measure nearness; here the metric becomes a tunable knob.
(The geometry of these metrics — how their neighbourhoods are shaped, and Mahalanobis distance — is
the subject of notebook 6.)

In [ ]:
print(f"{'metric':>16}   test accuracy")
for label, kw in (
    ("Manhattan (p=1)", {"p": 1}),
    ("Euclidean (p=2)", {"p": 2}),
    ("Chebyshev", {"metric": "chebyshev"}),
):
    model = KNeighborsClassifier(n_neighbors=15, **kw).fit(X_train, y_train)
    print(f"{label:>16}   {model.score(X_test, y_test):.3f}")

**Read the output.** The metric nudges accuracy a little — Manhattan edges ahead here (0.967 vs
Euclidean's 0.956, a single test point) — which echoes notebook 2's finding that the metric is a fine
adjustment next to the scale of the features. It is still worth trying: the best metric depends on the
data, so on a new problem you would treat `p` as one more hyperparameter to tune.

In [ ]:
# Choose `weights` honestly: cross-validation on the training set only (notebook 3's discipline).
for w in ("uniform", "distance"):
    cv = cross_val_score(
        KNeighborsClassifier(n_neighbors=15, weights=w), X_train, y_train, cv=skf
    )
    print(f"weights={w:9s}: CV accuracy {cv.mean():.3f}")

chosen = KNeighborsClassifier(n_neighbors=15, weights="distance").fit(X_train, y_train)
print(f"\nchosen (distance) confirmed once on the test set: {chosen.score(X_test, y_test):.3f}")

**Read the output.** Cross-validation on the training data prefers `distance` (0.924 vs 0.919 for
uniform) — a small margin, but a principled one, decided without touching the test set. The sealed
test set then confirms it: 0.967, up from uniform's 0.956. This is how you set any of these knobs
honestly — the discipline from notebook 3, now with the library's `cross_val_score` doing the work:
**cross-validation chooses on the training data; the test set confirms, once.**

## The even-k tie

With two classes and an **even** k, the vote can land in a dead heat: at k = 2, one neighbour of each
class. The estimator cannot return half a label — so how does it decide?

In [ ]:
knn2 = KNeighborsClassifier(n_neighbors=2).fit(X_train, y_train)
distances, indices = knn2.kneighbors(X_test)

shown = 0
for i in range(len(X_test)):
    labels = y_train[indices[i]]
    if labels[0] != labels[1]:  # the two neighbours disagree -> a 1-1 tie
        nearer = int(labels[np.argmin(distances[i])])
        pred = int(knn2.predict(X_test[i : i + 1])[0])
        print(f"neighbour labels {labels.tolist()}, nearer is class {nearer} -> predicts {pred}")
        shown += 1
        if shown == 4:
            break

**Read the output.** With `uniform` weights, scikit-learn breaks the tie by the **lowest class
label** — here, class 0 — and it does so every time, regardless of which neighbour is nearer. Look at
the example whose nearer neighbour is class 1: the prediction is still 0. The rule is deterministic
(you get the same answer every run), but it is arbitrary — it ignores proximity. Under the hood this
is an argmax over the per-class counts, which returns the first class in sorted order when they tie —
the same convention our by-hand `np.bincount(...).argmax()` used, which is why by-hand and sklearn
agreed earlier even without restricting to odd k. That is exactly why **odd k** is the default for two
classes: an odd number of votes cannot tie. (Had we used
`weights="distance"`, the nearer neighbour would have won the tie instead — a more sensible break, but
odd k sidesteps the question entirely.)

## Your turn

Reason these through.

1. `weights="distance"` barely changed accuracy at k = 15 but rescued it at k = 151. Why does
   distance-weighting matter more as k grows?
2. The "use odd k" rule avoids ties for **two** classes. With **three** classes, can an odd k still
   produce a tie among the neighbours? Give an example.
3. You move to a new dataset and wonder whether `p=1` or `p=2` is better. Describe exactly how you
   would decide — without looking at the test set.

## What you built

You graduated from the by-hand k-NN to scikit-learn's `KNeighborsClassifier` — and proved they are
the same rule before trusting the library. Then you toured its knobs: `n_neighbors` (the bias–variance
dial), `weights` (uniform versus distance, which rescues large-k k-NN from over-smoothing), and
`metric` (the Minkowski `p`, a fine dial). You chose `weights` by cross-validation and confirmed it
once on the test set, and you saw how the estimator breaks an even-k tie — deterministically, by the
lowest label — which is why odd k is the binary default.

**Vocabulary**

- **`KNeighborsClassifier`** — scikit-learn's k-NN estimator: the by-hand rule, optimized with a
  spatial index.
- **`n_neighbors`** — k, the bias–variance dial (notebook 3).
- **`weights`** — `uniform` (equal votes) or `distance` (votes weighted by 1 / distance).
- **`metric` / Minkowski `p`** — how distance is measured: `p=2` Euclidean, `p=1` Manhattan, and more.
- **deterministic tie-break** — with even k and uniform weights, a tie goes to the lowest class label.
- **tuning by `cross_val_score`** — choose a hyperparameter by CV on training data; confirm on test once.

## References

- scikit-learn, *`sklearn.neighbors.KNeighborsClassifier`* — User Guide §1.6 (Nearest Neighbors) and
  the API reference:
  <https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html>
- G. James, D. Witten, T. Hastie, R. Tibshirani (2021). *An Introduction to Statistical Learning*,
  §2.2.3 (k-NN). DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).
- T. M. Cover, P. E. Hart (1967). *Nearest neighbor pattern classification.* IEEE Trans. Inf. Theory
  13(1):21–27. DOI: [10.1109/TIT.1967.1053964](https://doi.org/10.1109/TIT.1967.1053964).

---

*Previous: 03 — The k dial* · *Next: 05 — Demanding case: breast cancer & the curse*